# SageMaker Workflow :: Stakeholder Question 1 

"**Predict whether a 311 complaint will be resolved within 3 days.**"

I am using this notebook to review and run a complete SageMaker **Linear Learner** workflow for this stakeholder question. This notebook is designed for a compare-and-contrast exercise/experiment so that I can connect the SageMaker built-in algorithm workflow to the sklearn work I completed previously.

At a high level, this notebook will:
- Load a modeling dataset from S3,
- Preprocess the data into the format Linear Learner expects,
- Upload train and test files back to S3,
- Launch a SageMaker training job,
- Use Batch Transform to generate predictions, and
- Evaluate the model output in the notebook.

# Setup

This notebook uses SageMaker's built-in **Linear Learner** algorithm to train a baseline model for this stakeholder question. The overall workflow is different from sklearn: I will still prepare data in the notebook, but the actual training job runs on separately managed compute resources in SageMaker and reads its input data from S3.

In the cells below, I will:
- Import the libraries needed for SageMaker, pandas, and evaluation,
- Connect to your SageMaker session and execution role,
- Define my S3 bucket and folder paths, and
- Retrieve the correct regional container image for Linear Learner.

In [1]:
import io
import boto3
import sagemaker
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = session.boto_region_name
s3client = boto3.client('s3')

print(f"Region: {region}")
print(f"Role: {role}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
Region: us-east-1
Role: arn:aws:iam::975049958391:role/LabRole


## S3 configuration

I am updating the bucket and prefix below so they match the location of my exported modeling CSV, which I extracted via Athena. Keeping a clear folder structure in S3 matters because the training data, test data, model artifacts, and prediction outputs are all written to separate locations rather than staying inside notebook memory as they would in sklearn.

In [3]:
S3_BUCKET = 'cmse492-monislow-nyc311-975049958391-us-east-1-an'
S3_PREFIX = 'modeling'

LL_IMAGE = sagemaker.image_uris.retrieve('linear-learner', region)

## Helper functions

Linear Learner expects a very specific CSV format: all features must be numeric, the label must be in the first column, and there should be no header row. These helper functions make it easier to upload properly formatted files to S3 and reload outputs such as batch prediction results.

In [4]:
def upload_csv_to_s3(df, bucket, key):
    """Upload a DataFrame as a headerless CSV to S3."""
    buf = io.BytesIO()
    df.to_csv(buf, index=False, header=False)
    buf.seek(0)
    s3client.put_object(Bucket=bucket, Key=key, Body=buf.getvalue())
    print(f"Uploaded s3://{bucket}/{key} with {len(df):,} rows")
    return f"s3://{bucket}/{key}"

def download_csv_from_s3(bucket, key, **kwargs):
    """Download a CSV from S3 into a DataFrame."""
    obj = s3client.get_object(Bucket=bucket, Key=key)
    data = obj['Body'].read().decode('utf-8')
    return pd.read_csv(io.StringIO(data), **kwargs)

def make_ll_dataframe(X, y):
    """
    Combine label (y) and features (X) into the format Linear Learner expects:
    label first, all numeric, no header.
    """
    if hasattr(X, 'toarray'):
        X = X.toarray()
    label_col = pd.Series(y, name='label').reset_index(drop=True)
    feat_df = pd.DataFrame(X).reset_index(drop=True)
    return pd.concat([label_col, feat_df], axis=1)

## Load and preprocess the data

This stakeholder question is a **binary classification** problem. The target column is `resolved_quickly`, where 1 means the complaint was resolved within 3 days and 0 means it took longer. This notebook keeps the same modeling logic we already used with sklearn, but it reformats the data into the structure required by Linear Learner.

The main preprocessing step here is one-hot encoding the categorical predictors. SageMaker built-in algorithms do not accept string columns directly, so columns like agency and borough must be converted to numeric indicators before training.

In [6]:
df = download_csv_from_s3(S3_BUCKET, f"{S3_PREFIX}/data_resolution_time/data_resolution_time.csv")
print(df.shape)
display(df.head())

# Features and target
cat_cols = ['agency', 'borough', 'problem_category']
num_cols = ['day_of_week', 'hour_of_day']
target = 'resolved_quickly'

X = df[cat_cols + num_cols]
y = df[target].astype(float)

# One-hot encode categoricals; pass numerics through
preprocessor = ColumnTransformer(
    transformers=[
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ('num', 'passthrough', num_cols),
    ]
)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Training class balance:')
print(pd.Series(y_train).value_counts(normalize=True).round(3))

(173870, 8)


,agency,borough,problem,incident_zip,day_of_week,hour_of_day,problem_category,resolved_quickly
0,HPD,BRONX,HEAT/HOT WATER,10460.0,4,8,housing,1
1,HPD,MANHATTAN,PAINT/PLASTER,10010.0,4,8,housing,1
2,NYPD,QUEENS,Noise - Vehicle,11372.0,4,8,other,1
3,DOT,BRONX,Traffic Signal Condition,10451.0,4,8,traffic,1
4,NYPD,QUEENS,Illegal Parking,11101.0,4,8,traffic,1


Train: (139096, 26) Test: (34774, 26)
Training class balance:
resolved_quickly
1.0    0.841
0.0    0.159
Name: proportion, dtype: float64


## Upload train and test files to S3

Even though I am working inside a notebook, SageMaker's managed training job cannot directly access variables sitting in notebook memory. That is why I convert the encoded arrays into headerless CSV files, place the label first, and upload the train and test data to S3 before training begins.

In [7]:
train_path = upload_csv_to_s3(
    make_ll_dataframe(X_train, y_train),
    S3_BUCKET,
    f"{S3_PREFIX}/ll-binary/train/train.csv",
)

test_path = upload_csv_to_s3(
    make_ll_dataframe(X_test, y_test),
    S3_BUCKET,
    f"{S3_PREFIX}/ll-binary/test/test.csv",
)

Uploaded s3://cmse492-monislow-nyc311-975049958391-us-east-1-an/modeling/ll-binary/train/train.csv with 139,096 rows
Uploaded s3://cmse492-monislow-nyc311-975049958391-us-east-1-an/modeling/ll-binary/test/test.csv with 34,774 rows


## Train the Linear Learner model

This cell defines the SageMaker estimator and submits the training job. Notice the key hyperparameter: `predictor_type='binary_classifier'` 

The code below sets the predictor type to binary classification, tells SageMaker how many encoded features exist, and lets SageMaker handle feature normalization internally. This is the main point where the workflow diverges from sklearn's `model.fit(...)` pattern.

In [8]:
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput

ll_model = Estimator(
    image_uri=LL_IMAGE,
    role=role,
    
    instance_count=1,
    instance_type='ml.m5.large',
    output_path=f"s3://{S3_BUCKET}/{S3_PREFIX}/ll-binary/output",
    sagemaker_session=session,
)

ll_model.set_hyperparameters(
    predictor_type='binary_classifier',
    feature_dim=X_train.shape[1],
    mini_batch_size=1000,
    epochs=10,
    normalize_data=True,
    normalize_label=False,
)

ll_model.fit(
    {'train': TrainingInput(train_path, content_type='text/csv')},
    logs=False, # Change this to True if you'd like to see the training output.
)

INFO:sagemaker:Creating training-job with name: linear-learner-2026-04-08-13-59-41-510



2026-04-08 13:59:41 Starting - Starting the training job..........................
2026-04-08 14:02:01 Starting - Preparing the instances for training....
2026-04-08 14:02:24 Downloading - Downloading input data.........
2026-04-08 14:03:14 Downloading - Downloading the training image..............
2026-04-08 14:04:30 Training - Training image download completed. Training in progress............
2026-04-08 14:05:31 Uploading - Uploading generated training model.
2026-04-08 14:05:44 Completed - Training job completed


## Run Batch Transform for predictions

Instead of deploying a real-time endpoint, this notebook uses **Batch Transform** to score the full test set. That is a good fit for experimentation because SageMaker spins up temporary inference compute, writes predictions back to S3, and then shuts the job down when it is finished. This avoids leaving a persistent endpoint running, which would consume resources and create additional costs.

In [9]:
transformer = ll_model.transformer(
    instance_count=1,
    instance_type='ml.m5.large',
    output_path=f"s3://{S3_BUCKET}/{S3_PREFIX}/ll-binary/predictions",
    accept='text/csv',
    assemble_with='Line',
)

# Run on the test set (features only — no label column)
test_features_path = upload_csv_to_s3(
    pd.DataFrame(X_test),
    S3_BUCKET,
    f"{S3_PREFIX}/ll-binary/test/test-features.csv",
)

transformer.transform(
    test_features_path,
    content_type='text/csv',
    split_type='Line',
    wait=True,
)

print('Batch transform complete.')

INFO:sagemaker:Creating model with name: linear-learner-2026-04-08-14-06-15-807
INFO:sagemaker:Creating transform job with name: linear-learner-2026-04-08-14-06-16-945


Uploaded s3://cmse492-monislow-nyc311-975049958391-us-east-1-an/modeling/ll-binary/test/test-features.csv with 34,774 rows
.........................................Docker entrypoint called with argument(s): serve
Running default environment configuration script
[04/08/2026 14:13:09 INFO 139839636584256] Memory profiler is not enabled by the environment variable ENABLE_PROFILER.
/opt/amazon/lib/python3.8/site-packages/mxnet/model.py:97: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if num_device is 1 and 'dist' not in kvstore:
/opt/amazon/lib/python3.8/site-packages/scipy/optimize/_shgo.py:495: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if cons['type'] is 'ineq':
/opt/amazon/lib/python3.8/site-packages/scipy/optimize/_shgo.py:743: SyntaxWarning: "is not" with a literal. Did you mean "!="?
  if len(self.X_min) is not 0:
[04/08/2026 14:13:13 WARNING 139839636584256] Loggers have already been setup.
[04/08/2026 14:13:13 INFO 139839636584256] loaded entry point clas

## Evaluate the predictions

The evaluation below mirrors the kind of performance review one does with sklearn, including overall accuracy, F1 score, and a full classification report.

In [10]:
# Download predictions and evaluate
pred_key = f"{S3_PREFIX}/ll-binary/predictions/test-features.csv.out"
preds = download_csv_from_s3(S3_BUCKET, pred_key, header=None)
display(preds.head())

y_pred = preds.iloc[:, 0].astype(float)

print('Binary classification results')
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 (class 1): {f1_score(y_test, y_pred):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=['Not quickly', 'Quickly']))

,0
0,1
1,1
2,1
3,1
4,1


Binary classification results
Accuracy: 0.8471
F1 (class 1): 0.9159

              precision    recall  f1-score   support

 Not quickly       0.64      0.09      0.16      5542
     Quickly       0.85      0.99      0.92     29232

    accuracy                           0.85     34774
   macro avg       0.74      0.54      0.54     34774
weighted avg       0.82      0.85      0.80     34774



## Wrap-up

After running this notebook, we can compare the results here to the sklearn workflow I used previously. As I review the outputs, I will have to pay attention to what changed conceptually: the model training happened as a SageMaker job, the data had to be staged in S3, and Batch Transform handled prediction without creating a persistent endpoint.

---
© Copyright 2026, The Department of Computational Mathematics, Science and Engineering at Michigan State University.